In [1]:
from functools import wraps
from typing import Callable, Any
from datetime import datetime as dt
import time

---

### Positional vs. Keyword arguments

In [2]:
def fun(a, b, c):
    return a + b - c

In [3]:
# Specifying arguments positionally
fun(1, 2, 3)

0

In [4]:
# Specifying arguments by keyword
fun(b=2, a=1, c=3)

0

In [5]:
# Mixture of both, positional arguments have to come first
fun(1, c=3, b=2)

0

---

### Operators for unpacking (`*` and `**`)

In [6]:
my_list = [1, 2, 3]
my_list

[1, 2, 3]

In [7]:
# Unpack the list and repackage with the number 4
my_list = [*my_list, 4]
my_list

[1, 2, 3, 4]

In [8]:
my_dict = {'a': 1, 'b': 2}
my_dict

{'a': 1, 'b': 2}

In [9]:
# Use ** to unpack and repack with a new element
my_dict = {**my_dict, 'c': 3}
my_dict

{'a': 1, 'b': 2, 'c': 3}

---

### A function that takes any number of positional and keyword arguments

In [10]:
def takes_anything(*args, **kwargs):
    '''
    Positional arguments will be packaged into a tuple called 'args.'
    Keyword arguments will be packaged into a dictionary called 'kwargs.'
    '''
    print(f'Positional arguments: {args}')
    print(f'Keyword arguments:  {kwargs}')
    return

In [11]:
takes_anything(42, 11, 3, 'cat', dog='arf', uhhh='ummm')

Positional arguments: (42, 11, 3, 'cat')
Keyword arguments:  {'dog': 'arf', 'uhhh': 'ummm'}


---

### Passing a function to a function

In [12]:
def name_of_function(func: Callable) -> None:
    print(f'The name of the function is: {func.__name__}')
    return

In [13]:
name_of_function(fun)

The name of the function is: fun


In [14]:
def list_args_and_run(func: Callable, *args, **kwargs) -> any:
    print(f'The name of the function is: {func.__name__}')
    print(f'Positional arguments: {args}')
    print(f'Keyword arguments: {kwargs}')
    print(f'Function was called at: {dt.now()}')
    return func(*args, **kwargs)

In [15]:
list_args_and_run(fun, 2, c=5, b=11)

The name of the function is: fun
Positional arguments: (2,)
Keyword arguments: {'c': 5, 'b': 11}
Function was called at: 2026-09-02 12:08:14.479103


8

---
### Decorators

A decorator is a function that takes a function as input and returns a modified version of the function. They have many uses, this one will be useful for debugging our code.

In [16]:
def log_calls_simple(func: Callable) -> Callable:
    def wrapper(*args, **kwargs) -> Any:
        print(f'{func.__name__} was called: {dt.now()}')
        print(f'Positional arguments: {args}')
        print(f'Keyword arguments: {kwargs}')

        t0 = dt.now()
        result = func(*args, **kwargs)
        runtime = dt.now() - t0
        
        print(f'{func.__name__} ran for: {runtime}')
        return result
    return wrapper

In [23]:
@log_calls_simple
def wait(secs: float) -> str:
    time.sleep(secs)
    return f'I waited for {secs} secs'

In [19]:
wait(3)

wait was called: 2026-09-02 12:15:04.437245
Positional arguments: (3,)
Keyword arguments: {}
wait ran for: 0:00:03.001401


'I waited for 3 secs'

In [21]:
decorated_wait = log_calls_simple(wait)

In [22]:
decorated_wait(2)

wrapper was called: 2026-09-02 12:16:21.155538
Positional arguments: (2,)
Keyword arguments: {}
wait was called: 2026-09-02 12:16:21.155669
Positional arguments: (2,)
Keyword arguments: {}
wait ran for: 0:00:02.005079
wrapper ran for: 0:00:02.005527


'I waited for 2 secs'

In [24]:
log_calls_simple(sum)

<function __main__.log_calls_simple.<locals>.wrapper(*args, **kwargs) -> Any>

In [26]:
sum

<function sum(iterable, /, start=0)>

---
### Decorators Factory

This is a function that returns a decorator. For this implementation, it will allow us to build some options into the decorator.

In [51]:
def log_calls(show_args: bool=True,
              show_runtime: bool=True
             ) -> Callable:
    '''
    Used to show positional and keyword arguments and print runtime.
    '''
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args, **kwargs) -> Any:
            '''
            I wrap functions.
            '''
            print(f'{func.__name__} was called: {dt.now()}')

            if show_args:
                print(f'\tPositional arguments: {args}')
                print(f'\tKeyword arguments: {kwargs}')
    
            t0 = dt.now()
            result = func(*args, **kwargs)
            runtime = dt.now() - t0

            if show_runtime:
                print(f'{func.__name__} ran for: {runtime}\n')
            return result
        return wrapper
    return decorator

In [52]:
@log_calls()
def wait_again(secs: float) -> str:
    '''
    I wait for a specified number of seconds.
    '''
    time.sleep(secs)
    return f'I waited for {secs} secs'

@log_calls(show_args=False)
def big_input_func(x: str) -> int:
    return len(x)

In [53]:
def main():
    x = 'abc'*100
    print(wait_again(2))
    print(big_input_func(x))

In [54]:
main()

wait_again was called: 2026-09-02 12:39:22.680698
	Positional arguments: (2,)
	Keyword arguments: {}
wait_again ran for: 0:00:02.005068

I waited for 2 secs
big_input_func was called: 2026-09-02 12:39:24.686220
big_input_func ran for: 0:00:00.000006

300
